<a href="https://colab.research.google.com/github/eltongaspar/python/blob/Advpl/Groq_Agemte_Especialista.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================================
# 📦 1. Instalar dependências
# =========================================
!apt-get install -y tesseract-ocr tesseract-ocr-por
!pip install groq gradio PyPDF2 pytesseract pillow pdf2image opencv-python numpy

In [ ]:
# =========================================
# 🔑 2. Imports + API Key
# =========================================
from google.colab import userdata
from groq import Groq
import gradio as gr
import time
import PyPDF2
import pytesseract
from PIL import Image
from pdf2image import convert_from_path
import mimetypes
import cv2
import numpy as np

api_key = userdata.get("GROQ_API_KEY")

if not api_key:
    raise ValueError("❌ API Key não encontrada nos Secrets")

client = Groq(api_key=api_key)

In [ ]:
# =========================================
# 📚 3. Contexto global
# =========================================
contexto_global = {"texto": ""}

In [ ]:
# =========================================
# 🔥 4. OCR MELHORADO
# =========================================
def ocr_imagem_melhorado(caminho):
    img = cv2.imread(caminho)

    # aumenta resolução
    img = cv2.resize(img, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)

    # escala de cinza
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # binarização
    _, thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY)

    # remoção de ruído
    kernel = np.ones((1,1), np.uint8)
    img_processada = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel)

    # OCR em português
    texto = pytesseract.image_to_string(img_processada, lang="por")

    return texto

In [ ]:
# =========================================
# 📄 5. Leitura de arquivos
# =========================================
def ler_arquivo(file):
    if file is None:
        return ""

    texto = ""

    try:
        print("📂 Arquivo:", file.name)

        tipo, _ = mimetypes.guess_type(file.name)

        # TXT
        if tipo == "text/plain":
            with open(file.name, "r", encoding="utf-8") as f:
                texto = f.read()

        # PDF
        elif tipo == "application/pdf":
            with open(file.name, "rb") as f:
                reader = PyPDF2.PdfReader(f)
                for page in reader.pages:
                    texto += page.extract_text() or ""

            # OCR se necessário
            if not texto.strip():
                print("🔎 PDF escaneado → OCR")
                imagens = convert_from_path(file.name)
                for img in imagens:
                    caminho_temp = "/tmp/temp.png"
                    img.save(caminho_temp)
                    texto += ocr_imagem_melhorado(caminho_temp)

        # IMAGEM
        elif tipo and tipo.startswith("image"):
            print("🖼️ OCR em imagem")
            texto = ocr_imagem_melhorado(file.name)

        else:
            # fallback manual
            nome = file.name.lower()
            if any(ext in nome for ext in [".jpg", ".jpeg", ".png"]):
                texto = ocr_imagem_melhorado(file.name)

    except Exception as e:
        print("❌ Erro:", e)
        return ""

    return texto[:5000]

# =========================================
# 📤 Upload
# =========================================
def upload_arquivo(file):
    texto = ler_arquivo(file)

    if not texto.strip():
        return "❌ Não foi possível ler o arquivo"

    contexto_global["texto"] = texto
    return "✅ Arquivo carregado com sucesso!"

In [ ]:
# =========================================
# 🤖 6. Agente IA
# =========================================
class AgenteIA:
    def __init__(self, nome, especialidades):
        self.nome = nome
        self.especialidades = especialidades
        self.modelos = [
            "llama-3.3-70b-versatile",
            "llama-3.1-8b-instant"
        ]

    def gerar_prompt(self):
        return f"""
        Você é {self.nome}, especialista em {", ".join(self.especialidades)}.
        Explique de forma didática e prática.
        """

    def responder(self, pergunta, historico):
        for modelo in self.modelos:
            try:
                inicio = time.time()

                contexto = contexto_global["texto"]

                messages = [{
                    "role": "system",
                    "content": self.gerar_prompt() + f"\n\nMaterial de apoio:\n{contexto}"
                }]

                for h in historico:
                    messages.append({"role": "user", "content": h[0]})
                    messages.append({"role": "assistant", "content": h[1]})

                messages.append({"role": "user", "content": pergunta})

                response = client.chat.completions.create(
                    model=modelo,
                    messages=messages,
                    temperature=0.7,
                )

                tempo = round(time.time() - inicio, 2)
                resposta = response.choices[0].message.content

                return f"""⏱️ {tempo}s | 🧠 {modelo}

{resposta}"""

            except Exception as e:
                if "model_decommissioned" in str(e):
                    continue
                return f"❌ Erro: {e}"

        return "❌ Nenhum modelo disponível."

In [ ]:
# =========================================
# 🎯 7. Criar agente
# =========================================
agente_global = {"instancia": None}

def criar_agente(nome, especialidades):
    lista = [e.strip() for e in especialidades.split(",") if e.strip()]

    if not lista:
        return "❌ Informe especialidades"

    agente_global["instancia"] = AgenteIA(nome, lista)

    return f"✅ Agente criado: {nome}"

In [ ]:
# =========================================
# 💬 8. Chat
# =========================================
def chat(mensagem, historico):
    agente = agente_global["instancia"]

    if agente is None:
        historico.append([mensagem, "❌ Crie o agente primeiro"])
        return historico, ""

    resposta = agente.responder(mensagem, historico)
    historico.append([mensagem, resposta])

    return historico, ""

In [ ]:
# =========================================
# 🎨 9. Interface
# =========================================
with gr.Blocks() as app:
    gr.Markdown("# 🤖 Agente IA com OCR Avançado")

    with gr.Row():
        nome = gr.Textbox(label="Nome do agente")
        especialidades = gr.Textbox(label="Especialidades")

    btn_criar = gr.Button("Criar Agente")
    status = gr.Textbox()

    gr.Markdown("## 📂 Upload")
    arquivo = gr.File()
    btn_upload = gr.Button("Carregar")
    status_upload = gr.Textbox()

    chatbot = gr.Chatbot()
    msg = gr.Textbox()
    enviar = gr.Button("Enviar")

    btn_criar.click(criar_agente, [nome, especialidades], status)
    btn_upload.click(upload_arquivo, arquivo, status_upload)

    enviar.click(chat, [msg, chatbot], [chatbot, msg])
    msg.submit(chat, [msg, chatbot], [chatbot, msg])

# =========================================
# 🌐 10. Rodar
# =========================================
app.launch(share=True)

In [ ]:
# =========================================
# 📦 Instalação
# =========================================
!apt-get install -y tesseract-ocr tesseract-ocr-por
!pip install groq gradio pytesseract pillow opencv-python numpy PyPDF2 pdf2image google-cloud-vision

# =========================================
# 🔑 Imports
# =========================================
from google.colab import userdata
from groq import Groq
import gradio as gr
import pytesseract
import cv2
import numpy as np
import time
import os
import PyPDF2
from pdf2image import convert_from_path
from google.cloud import vision

# =========================================
# 🔐 CONFIG
# =========================================
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/content/cred.json"

try:
    vision_client = vision.ImageAnnotatorClient()
except:
    vision_client = None

api_key = userdata.get("GROQ_API_KEY")
groq_client = Groq(api_key=api_key) if api_key else None

# =========================================
# 🧠 OCR GOOGLE
# =========================================
def ocr_google(path):
    if not vision_client:
        return ""

    try:
        with open(path, "rb") as f:
            content = f.read()

        image = vision.Image(content=content)
        response = vision_client.text_detection(image=image)

        texts = response.text_annotations
        return texts[0].description if texts else ""

    except:
        return ""

# =========================================
# 🔁 OCR FALLBACK
# =========================================
def ocr_tesseract(path):
    img = cv2.imread(path)
    if img is None:
        return ""

    img = cv2.resize(img, None, fx=2, fy=2)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY)

    return pytesseract.image_to_string(thresh, lang="por")

def ocr_total(path):
    texto = ocr_google(path)
    if not texto.strip():
        texto = ocr_tesseract(path)
    return texto

# =========================================
# 📄 LEITOR UNIVERSAL
# =========================================
def ler_arquivo(file):
    if file is None:
        return ""

    path = file.name

    # TXT
    if path.endswith(".txt"):
        with open(path, "r", encoding="utf-8") as f:
            return f.read()

    # PDF
    elif path.endswith(".pdf"):
        texto = ""

        with open(path, "rb") as f:
            reader = PyPDF2.PdfReader(f)
            for page in reader.pages:
                texto += page.extract_text() or ""

        # OCR se PDF escaneado
        if not texto.strip():
            imagens = convert_from_path(path)
            for img in imagens:
                temp = "/tmp/page.png"
                img.save(temp)
                texto += ocr_total(temp)

        return texto

    # IMAGEM
    else:
        return ocr_total(path)

# =========================================
# 🤖 IA
# =========================================
def responder(pergunta, contexto):
    if not groq_client:
        return "❌ API não configurada"

    inicio = time.time()

    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "Você é um professor de TI. Explique de forma didática."},
            {"role": "user", "content": f"Material:\n{contexto}\n\nPergunta:\n{pergunta}"}
        ]
    )

    tempo = round(time.time() - inicio, 2)
    return f"⏱️ {tempo}s\n\n{response.choices[0].message.content}"

# =========================================
# 📂 PROCESSAMENTO
# =========================================
contexto_global = {"texto": ""}

def processar(file):
    if file is None:
        return None, "", ""

    texto = ler_arquivo(file)
    contexto_global["texto"] = texto

    return file.name, texto[:5000], ""

# =========================================
# 💬 CHAT
# =========================================
def chat(pergunta, historico):
    contexto = contexto_global["texto"]

    if not contexto:
        resposta = "❌ Envie um arquivo primeiro"
    else:
        resposta = responder(pergunta, contexto)

    historico.append((pergunta, resposta))
    return historico, ""

# =========================================
# 🎨 INTERFACE
# =========================================
with gr.Blocks() as app:
    gr.Markdown("# 🤖 IA com Leitura de Arquivos (Imagem + Texto + PDF)")

    with gr.Row():
        arquivo = gr.File(label="📂 Upload (imagem, PDF ou TXT)")
        btn = gr.Button("Processar")

    imagem = gr.Image(label="🖼️ Visual")
    texto = gr.Textbox(label="📄 Conteúdo extraído", lines=15)

    chatbot = gr.Chatbot()
    msg = gr.Textbox(label="Pergunta")

    btn.click(processar, arquivo, [imagem, texto])
    msg.submit(chat, [msg, chatbot], [chatbot, msg])

# =========================================
# 🚀 RODAR
# =========================================
app.launch(share=True)